In [ ]:
# STM32F429 GPIO (General Purpose Input/Output) Tutorial

This comprehensive tutorial covers the GPIO peripheral on the STM32F429 microcontroller. GPIO pins provide flexible digital input/output capabilities essential for interfacing with external devices, sensors, actuators, and communication interfaces.

## Table of Contents
1. [GPIO Overview](#gpio-overview)
2. [Hardware Architecture](#hardware-architecture)
3. [Register Configuration](#register-configuration)
4. [API Reference](#api-reference)
5. [Basic Usage Examples](#basic-usage-examples)
6. [Advanced Features](#advanced-features)
7. [Pin Functions](#pin-functions)
8. [Performance Considerations](#performance-considerations)
9. [Troubleshooting](#troubleshooting)

---

## GPIO Overview

The STM32F429 GPIO peripheral provides 114 general-purpose I/O pins organized into 9 ports (GPIOA through GPIOI), each with up to 16 pins. GPIO pins support multiple functions including digital input/output, analog input, and alternate functions for peripherals.

### Key Features
- **114 I/O Pins**: Organized in 9 ports (A-I) with up to 16 pins each
- **Multiple Modes**: Input, output, alternate function, and analog modes
- **Flexible Configuration**: Programmable speed, pull-up/down resistors, and output type
- **Interrupt Capability**: External interrupt lines for input pins
- **Alternate Functions**: Pin multiplexing for peripheral functions
- **Bit-Banding**: Atomic bit operations for thread-safe access
- **Lock Mechanism**: Pin configuration locking for security
- **High Current Drive**: Up to 25mA per pin with current limiting

### Applications
- Digital input/output control
- LED and display driving
- Button and switch interfacing
- Sensor connections
- Motor and relay control
- Communication interfaces (SPI, I2C, UART)
- PWM signal generation
- Analog signal conditioning
- Interrupt-driven event handling
- Power management control

### Performance Benefits
- **High Speed**: Up to 180MHz toggle frequency
- **Low Latency**: Direct register access with minimal overhead
- **Atomic Operations**: Bit-banding for thread-safe modifications
- **Flexible Drive**: Configurable output drive strength
- **Interrupt Response**: Fast external interrupt handling
- **Power Efficient**: Configurable pull-up/down and output modes
- **Robust Design**: ESD protection and overcurrent limiting

### GPIO Operating Modes
- **Input Modes**: Floating, pull-up, pull-down, analog
- **Output Modes**: Push-pull, open-drain, with configurable speed
- **Alternate Function**: Peripheral pin multiplexing
- **Analog Mode**: ADC/DAC signal routing
- **Interrupt Modes**: Rising/falling edge, high/low level triggers

---

## Hardware Architecture

### STM32F429 GPIO Block Diagram

```
AHB1 Bus (168MHz)
   |
   v
   +---------------------+
   |   GPIO Controller   |
   |                     |
   |  +---------------+  |
   |  |   Port A      |  |----> PA[0:15] Pins
   |  |   (16 pins)   |  |
   |  +---------------+  |
   |         |           |
   |  +---------------+  |----> PB[0:15] Pins
   |  |   Port B      |  |
   |  |   (16 pins)   |  |
   |  +---------------+  |
   |         |           |
   |  ...             |  |
   |         |           |
   |  +---------------+  |----> PI[0:11] Pins
   |  |   Port I      |  |
   |  |   (12 pins)   |  |
   |  +---------------+  |
   +---------------------+
             |
             | External Pins
             v
   +---------------------+
   |  I/O Pad Structure |
   |                     |
   |  ESD Protection    |
   |  Pull-up/down      |
   |  Output Driver     |
   |  Input Buffer      |
   |  Analog Switch     |
   +---------------------+
```

### Key Components
- **Port Registers**: Configuration and data registers for each port
- **Pin Multiplexer**: Alternate function selection
- **Output Drivers**: Push-pull and open-drain output stages
- **Input Buffers**: Digital input with configurable thresholds
- **Pull-up/down**: Internal resistor configuration
- **ESD Protection**: Electrostatic discharge protection circuits
- **Analog Switches**: Signal routing for ADC/DAC functions
- **Interrupt Controller**: External interrupt generation

### GPIO Ports and Pins
- **GPIOA-GPIOH**: 16 pins each (PA0-PA15 through PH0-PH15)
- **GPIOI**: 12 pins (PI0-PI11)
- **Total**: 114 I/O pins across 9 ports

### Pin Structure
Each GPIO pin includes:
- **Output Driver**: Push-pull or open-drain with configurable speed
- **Input Buffer**: Digital input with Schmitt trigger
- **Pull-up/down Resistors**: Internal 40kΩ resistors
- **Analog Switch**: For ADC/DAC connectivity
- **Protection Diodes**: ESD protection up to ±2kV
- **Current Limiting**: Overcurrent protection

### Alternate Functions
Each pin can be configured for alternate functions:
- **AF0**: System functions (JTAG, SWD, RTC)
- **AF1-AF15**: Peripheral functions (SPI, I2C, UART, etc.)
- **Pin Mapping**: Configurable through AFR registers
- **Function Selection**: AFR[0] for pins 0-7, AFR[1] for pins 8-15

---

## Register Configuration

### GPIO Control Registers Overview

The STM32F429 GPIO peripheral uses multiple registers per port for complete pin control:

#### GPIOx_MODER (Mode Register) - 0x4002 0000 + port_offset
- **Purpose**: Configures the mode for each pin (input/output/analog/alternate)
- **Access**: Read/Write
- **Reset Value**: 0xA800 0000 (varies by pin)
- **Bit Fields**: 2 bits per pin (MODERy[1:0])
  - 00: Input mode
  - 01: General purpose output mode
  - 10: Alternate function mode
  - 11: Analog mode

#### GPIOx_OTYPER (Output Type Register) - 0x4002 0004 + port_offset
- **Purpose**: Configures output type (push-pull or open-drain)
- **Access**: Read/Write
- **Reset Value**: 0x0000 0000
- **Bit Fields**: 1 bit per pin (OTy)
  - 0: Push-pull
  - 1: Open-drain

#### GPIOx_OSPEEDR (Output Speed Register) - 0x4002 0008 + port_offset
- **Purpose**: Configures output speed for each pin
- **Access**: Read/Write
- **Reset Value**: 0x0000 0000
- **Bit Fields**: 2 bits per pin (OSPEEDRy[1:0])
  - 00: Low speed (2MHz)
  - 01: Medium speed (25MHz)
  - 10: Fast speed (50MHz)
  - 11: High speed (100MHz)

#### GPIOx_PUPDR (Pull-up/Pull-down Register) - 0x4002 000C + port_offset
- **Purpose**: Configures pull-up/pull-down resistors
- **Access**: Read/Write
- **Reset Value**: 0x0000 0000
- **Bit Fields**: 2 bits per pin (PUPDRy[1:0])
  - 00: No pull-up, pull-down
  - 01: Pull-up
  - 10: Pull-down
  - 11: Reserved

#### GPIOx_IDR (Input Data Register) - 0x4002 0010 + port_offset
- **Purpose**: Reads the input value of each pin
- **Access**: Read-only
- **Reset Value**: Undefined
- **Bit Fields**: 1 bit per pin (IDRy)

#### GPIOx_ODR (Output Data Register) - 0x4002 0014 + port_offset
- **Purpose**: Sets the output value of each pin
- **Access**: Read/Write
- **Reset Value**: 0x0000 0000
- **Bit Fields**: 1 bit per pin (ODRy)

#### GPIOx_BSRR (Bit Set/Reset Register) - 0x4002 0018 + port_offset
- **Purpose**: Atomic bit set/reset operations
- **Access**: Write-only
- **Reset Value**: N/A
- **Bit Fields**:
  - BSy (bits 0-15): Set pin y
  - BRy (bits 16-31): Reset pin y

#### GPIOx_LCKR (Lock Register) - 0x4002 001C + port_offset
- **Purpose**: Locks pin configuration
- **Access**: Read/Write
- **Reset Value**: 0x0000 0000
- **Bit Fields**:
  - LCKy (bits 0-15): Lock pin y
  - LCKK: Lock key

#### GPIOx_AFRL/AFRH (Alternate Function Registers) - 0x4002 0020/0024 + port_offset
- **Purpose**: Configures alternate functions for pins
- **Access**: Read/Write
- **Reset Value**: 0x0000 0000
- **Bit Fields**: 4 bits per pin (AFRLy[3:0] for pins 0-7, AFRHy[3:0] for pins 8-15)

### Register Setup Sequence

**Basic Output Pin Configuration:**
```c
// Enable GPIOA clock
RCC->AHB1ENR |= RCC_AHB1ENR_GPIOAEN;

// Configure PA5 as push-pull output, high speed, no pull
GPIOA->MODER |= GPIO_MODER_MODER5_0;        // Output mode
GPIOA->OTYPER &= ~GPIO_OTYPER_OT_5;         // Push-pull
GPIOA->OSPEEDR |= GPIO_OSPEEDER_OSPEEDR5;   // High speed
GPIOA->PUPDR &= ~GPIO_PUPDR_PUPDR5;         // No pull

// Set output high
GPIOA->BSRR = GPIO_BSRR_BS_5;
```

**Input Pin with Interrupt Configuration:**
```c
// Configure PC13 as input with pull-up and interrupt
GPIOC->MODER &= ~GPIO_MODER_MODER13;        // Input mode
GPIOC->PUPDR |= GPIO_PUPDR_PUPDR13_0;       // Pull-up

// Configure external interrupt
SYSCFG->EXTICR[3] |= SYSCFG_EXTICR4_EXTI13_PC;  // PC13 source
EXTI->IMR |= EXTI_IMR_MR13;                  // Unmask interrupt
EXTI->FTSR |= EXTI_FTSR_TR13;                // Falling edge trigger

// Enable interrupt in NVIC
NVIC_EnableIRQ(EXTI15_10_IRQn);
```

**Alternate Function Configuration:**
```c
// Configure PA9/PA10 for UART1 TX/RX
GPIOA->MODER |= GPIO_MODER_MODER9_1 | GPIO_MODER_MODER10_1;  // Alternate function
GPIOA->AFR[1] |= (7 << GPIO_AFRH_AFRH1_Pos) | (7 << GPIO_AFRH_AFRH2_Pos);  // AF7 for UART1
GPIOA->OSPEEDR |= GPIO_OSPEEDER_OSPEEDR9 | GPIO_OSPEEDER_OSPEEDR10;  // High speed
```

---

## API Reference

### Configuration Structures

```c
typedef struct {
    uint32_t Pin;        // GPIO_PIN_0 through GPIO_PIN_15
    uint32_t Mode;       // GPIO_MODE_INPUT/OUTPUT/ALTERNATE/ANALOG
    uint32_t Pull;       // GPIO_NOPULL/PULLUP/PULLDOWN
    uint32_t Speed;      // GPIO_SPEED_LOW/MEDIUM/FAST/HIGH
    uint32_t Alternate;  // GPIO_AF0 through GPIO_AF15
} GPIO_InitTypeDef;

typedef struct {
    GPIO_TypeDef *Port;  // GPIOA through GPIOI
    GPIO_InitTypeDef Init;
    bool initialized;
    uint32_t errorCode;
} GPIO_Driver_Handle_t;
```

### Core Functions

#### Initialization Functions
```c
HAL_StatusTypeDef GPIO_Driver_Init(GPIO_Driver_Handle_t *handle);
HAL_StatusTypeDef GPIO_Driver_DeInit(GPIO_Driver_Handle_t *handle);
HAL_StatusTypeDef GPIO_Driver_Pin_Init(GPIO_TypeDef *GPIOx, GPIO_InitTypeDef *GPIO_Init);
```

#### Digital I/O Functions
```c
GPIO_PinState GPIO_Driver_ReadPin(GPIO_TypeDef *GPIOx, uint16_t GPIO_Pin);
void GPIO_Driver_WritePin(GPIO_TypeDef *GPIOx, uint16_t GPIO_Pin, GPIO_PinState PinState);
void GPIO_Driver_TogglePin(GPIO_TypeDef *GPIOx, uint16_t GPIO_Pin);
GPIO_PinState GPIO_Driver_ReadPort(GPIO_TypeDef *GPIOx);
void GPIO_Driver_WritePort(GPIO_TypeDef *GPIOx, uint16_t PortValue);
```

#### Interrupt Functions
```c
HAL_StatusTypeDef GPIO_Driver_EnableIT(GPIO_TypeDef *GPIOx, uint16_t GPIO_Pin, uint32_t edge);
HAL_StatusTypeDef GPIO_Driver_DisableIT(GPIO_TypeDef *GPIOx, uint16_t GPIO_Pin);
void GPIO_Driver_ClearITPendingBit(uint16_t GPIO_Pin);
```

#### Advanced Functions
```c
HAL_StatusTypeDef GPIO_Driver_LockPin(GPIO_TypeDef *GPIOx, uint16_t GPIO_Pin);
uint32_t GPIO_Driver_GetError(GPIO_Driver_Handle_t *handle);
```

### Constants and Macros

```c
// Pin definitions
#define GPIO_PIN_0                 ((uint16_t)0x0001)
#define GPIO_PIN_1                 ((uint16_t)0x0002)
#define GPIO_PIN_2                 ((uint16_t)0x0004)
#define GPIO_PIN_3                 ((uint16_t)0x0008)
#define GPIO_PIN_4                 ((uint16_t)0x0010)
#define GPIO_PIN_5                 ((uint16_t)0x0020)
#define GPIO_PIN_6                 ((uint16_t)0x0040)
#define GPIO_PIN_7                 ((uint16_t)0x0080)
#define GPIO_PIN_8                 ((uint16_t)0x0100)
#define GPIO_PIN_9                 ((uint16_t)0x0200)
#define GPIO_PIN_10                ((uint16_t)0x0400)
#define GPIO_PIN_11                ((uint16_t)0x0800)
#define GPIO_PIN_12                ((uint16_t)0x1000)
#define GPIO_PIN_13                ((uint16_t)0x2000)
#define GPIO_PIN_14                ((uint16_t)0x4000)
#define GPIO_PIN_15                ((uint16_t)0x8000)
#define GPIO_PIN_ALL               ((uint16_t)0xFFFF)

// Mode definitions
#define GPIO_MODE_INPUT            0x00000000U
#define GPIO_MODE_OUTPUT_PP        0x00000001U
#define GPIO_MODE_OUTPUT_OD        0x00000011U
#define GPIO_MODE_AF_PP            0x00000002U
#define GPIO_MODE_AF_OD            0x00000012U
#define GPIO_MODE_ANALOG           0x00000003U
#define GPIO_MODE_IT_RISING        0x10110000U
#define GPIO_MODE_IT_FALLING       0x10210000U
#define GPIO_MODE_IT_RISING_FALLING 0x10310000U
#define GPIO_MODE_EVT_RISING       0x10120000U
#define GPIO_MODE_EVT_FALLING      0x10220000U
#define GPIO_MODE_EVT_RISING_FALLING 0x10320000U

// Speed definitions
#define GPIO_SPEED_LOW             0x00000000U
#define GPIO_SPEED_MEDIUM          0x00000001U
#define GPIO_SPEED_FAST            0x00000002U
#define GPIO_SPEED_HIGH            0x00000003U

// Pull definitions
#define GPIO_NOPULL                0x00000000U
#define GPIO_PULLUP                0x00000001U
#define GPIO_PULLDOWN              0x00000002U

// Alternate function definitions
#define GPIO_AF0_SYSTEM            0x00000000U
#define GPIO_AF1_TIM1              0x00000001U
#define GPIO_AF2_TIM3              0x00000002U
#define GPIO_AF3_TIM8              0x00000003U
#define GPIO_AF4_I2C1              0x00000004U
#define GPIO_AF5_SPI1              0x00000005U
#define GPIO_AF6_SPI3              0x00000006U
#define GPIO_AF7_USART1            0x00000007U
#define GPIO_AF8_USART6            0x00000008U
#define GPIO_AF9_I2C2              0x00000009U
#define GPIO_AF10_OTG_FS           0x0000000AU
#define GPIO_AF11_ETH              0x0000000BU
#define GPIO_AF12_FMC              0x0000000CU
#define GPIO_AF13_DCMI             0x0000000DU
#define GPIO_AF14_LTDC             0x0000000EU
#define GPIO_AF15_EVENTOUT         0x0000000FU

// Error codes
#define GPIO_DRIVER_ERROR_NONE     0x00U
#define GPIO_DRIVER_ERROR_INIT     0x01U
#define GPIO_DRIVER_ERROR_CONFIG   0x02U
#define GPIO_DRIVER_ERROR_PIN      0x04U
```

### Error Codes

| Error Code | Description |
|------------|-------------|
| GPIO_DRIVER_ERROR_NONE | Operation successful |
| GPIO_DRIVER_ERROR_INIT | Initialization failed |
| GPIO_DRIVER_ERROR_CONFIG | Configuration error |
| GPIO_DRIVER_ERROR_PIN | Invalid pin specification |
| HAL_GPIO_ERROR_NONE | GPIO operation OK |
| HAL_GPIO_ERROR_INVALID_PARAM | Invalid parameter |

---

## Basic Usage Examples

### Example 1: LED Control

```c
#include "Peripherals/GPIO/gpio.h"

int main(void) {
    // Initialize system
    HAL_Init();
    
    // Configure LED pin (PA5)
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Pin = GPIO_PIN_5,
        .Mode = GPIO_MODE_OUTPUT_PP,
        .Pull = GPIO_NOPULL,
        .Speed = GPIO_SPEED_HIGH
    };
    
    GPIO_Driver_Handle_t gpioHandle = {
        .Port = GPIOA,
        .Init = GPIO_InitStruct
    };
    
    // Initialize GPIO
    if (GPIO_Driver_Init(&gpioHandle) != HAL_OK) {
        // Handle initialization error
        while(1);
    }
    
    // Blink LED
    while (1) {
        GPIO_Driver_TogglePin(GPIOA, GPIO_PIN_5);
        HAL_Delay(500);  // 500ms delay
    }
}
```

### Example 2: Button Input with Polling

```c
// Configure button input pin (PC13)
GPIO_InitTypeDef buttonInit = {
    .Pin = GPIO_PIN_13,
    .Mode = GPIO_MODE_INPUT,
    .Pull = GPIO_PULLUP,
    .Speed = GPIO_SPEED_LOW
};

GPIO_Driver_Handle_t buttonHandle = {
    .Port = GPIOC,
    .Init = buttonInit
};

// Initialize button GPIO
GPIO_Driver_Init(&buttonHandle);

// Main loop
while (1) {
    // Read button state (active low)
    if (GPIO_Driver_ReadPin(GPIOC, GPIO_PIN_13) == GPIO_PIN_RESET) {
        // Button pressed
        GPIO_Driver_WritePin(GPIOA, GPIO_PIN_5, GPIO_PIN_SET);  // Turn on LED
        HAL_Delay(100);  // Debounce delay
        
        // Wait for button release
        while (GPIO_Driver_ReadPin(GPIOC, GPIO_PIN_13) == GPIO_PIN_RESET);
        HAL_Delay(100);  // Debounce delay
        
        GPIO_Driver_WritePin(GPIOA, GPIO_PIN_5, GPIO_PIN_RESET);  // Turn off LED
    }
}
```

### Example 3: UART Pin Configuration

```c
// Configure UART1 pins (PA9 TX, PA10 RX)
void configure_uart1_pins(void) {
    GPIO_InitTypeDef GPIO_InitStruct;
    
    // Enable GPIOA clock
    __HAL_RCC_GPIOA_CLK_ENABLE();
    
    // Configure TX pin (PA9)
    GPIO_InitStruct.Pin = GPIO_PIN_9;
    GPIO_InitStruct.Mode = GPIO_MODE_AF_PP;
    GPIO_InitStruct.Pull = GPIO_PULLUP;
    GPIO_InitStruct.Speed = GPIO_SPEED_HIGH;
    GPIO_InitStruct.Alternate = GPIO_AF7_USART1;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // Configure RX pin (PA10)
    GPIO_InitStruct.Pin = GPIO_PIN_10;
    GPIO_InitStruct.Mode = GPIO_MODE_AF_PP;
    GPIO_InitStruct.Pull = GPIO_PULLUP;
    GPIO_InitStruct.Speed = GPIO_SPEED_HIGH;
    GPIO_InitStruct.Alternate = GPIO_AF7_USART1;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
}
```

### Example 4: SPI Pin Configuration

```c
// Configure SPI1 pins (PA5 SCK, PA6 MISO, PA7 MOSI)
void configure_spi1_pins(void) {
    GPIO_InitTypeDef GPIO_InitStruct;
    
    // Enable GPIOA clock
    __HAL_RCC_GPIOA_CLK_ENABLE();
    
    // Configure SCK pin (PA5)
    GPIO_InitStruct.Pin = GPIO_PIN_5;
    GPIO_InitStruct.Mode = GPIO_MODE_AF_PP;
    GPIO_InitStruct.Pull = GPIO_NOPULL;
    GPIO_InitStruct.Speed = GPIO_SPEED_HIGH;
    GPIO_InitStruct.Alternate = GPIO_AF5_SPI1;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // Configure MISO pin (PA6)
    GPIO_InitStruct.Pin = GPIO_PIN_6;
    GPIO_InitStruct.Mode = GPIO_MODE_AF_PP;
    GPIO_InitStruct.Pull = GPIO_NOPULL;
    GPIO_InitStruct.Speed = GPIO_SPEED_HIGH;
    GPIO_InitStruct.Alternate = GPIO_AF5_SPI1;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // Configure MOSI pin (PA7)
    GPIO_InitStruct.Pin = GPIO_PIN_7;
    GPIO_InitStruct.Mode = GPIO_MODE_AF_PP;
    GPIO_InitStruct.Pull = GPIO_NOPULL;
    GPIO_InitStruct.Speed = GPIO_SPEED_HIGH;
    GPIO_InitStruct.Alternate = GPIO_AF5_SPI1;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
}
```

### Example 5: External Interrupt Configuration

```c
// Configure external interrupt on PC13 (user button)
void configure_exti_button(void) {
    GPIO_InitTypeDef GPIO_InitStruct;
    
    // Enable GPIOC clock
    __HAL_RCC_GPIOC_CLK_ENABLE();
    
    // Configure PC13 as input with interrupt
    GPIO_InitStruct.Pin = GPIO_PIN_13;
    GPIO_InitStruct.Mode = GPIO_MODE_IT_FALLING;
    GPIO_InitStruct.Pull = GPIO_PULLUP;
    HAL_GPIO_Init(GPIOC, &GPIO_InitStruct);
    
    // Configure interrupt priority
    HAL_NVIC_SetPriority(EXTI15_10_IRQn, 0, 0);
    HAL_NVIC_EnableIRQ(EXTI15_10_IRQn);
}

// Interrupt handler
void EXTI15_10_IRQHandler(void) {
    if (__HAL_GPIO_EXTI_GET_IT(GPIO_PIN_13) != RESET) {
        __HAL_GPIO_EXTI_CLEAR_IT(GPIO_PIN_13);
        
        // Handle button press
        GPIO_Driver_TogglePin(GPIOA, GPIO_PIN_5);  // Toggle LED
    }
}
```

---

## Advanced Features

### Bit-Banding for Atomic Operations

```c
// Bit-banding allows atomic bit operations
#define BITBAND_SRAM_REF   0x20000000
#define BITBAND_SRAM_BASE  0x22000000
#define BITBAND_PERIPH_REF 0x40000000
#define BITBAND_PERIPH_BASE 0x42000000

// Convert SRAM address to bit-band address
#define BITBAND_SRAM(addr, bit) ((BITBAND_SRAM_BASE + (((uint32_t)addr) - BITBAND_SRAM_REF) * 32 + (bit) * 4))

// Convert peripheral address to bit-band address
#define BITBAND_PERIPH(addr, bit) ((BITBAND_PERIPH_BASE + (((uint32_t)addr) - BITBAND_PERIPH_REF) * 32 + (bit) * 4))

// Atomic GPIO operations using bit-banding
void atomic_gpio_operations(void) {
    // Atomic set/clear operations
    volatile uint32_t *gpio_bsrr_bs5 = (uint32_t*)BITBAND_PERIPH(&GPIOA->BSRR, 5);
    volatile uint32_t *gpio_bsrr_br5 = (uint32_t*)BITBAND_PERIPH(&GPIOA->BSRR, 21);  // BS5 is bit 5, BR5 is bit 21
    
    // Atomic set pin
    *gpio_bsrr_bs5 = 1;
    
    // Atomic clear pin
    *gpio_bsrr_br5 = 1;
    
    // Atomic read operation
    volatile uint32_t *gpio_idr_5 = (uint32_t*)BITBAND_PERIPH(&GPIOA->IDR, 5);
    uint32_t pin_state = *gpio_idr_5;
}
```

### Pin Locking for Security

```c
// Lock GPIO configuration to prevent accidental changes
HAL_StatusTypeDef lock_gpio_pin(GPIO_TypeDef *GPIOx, uint16_t pin) {
    // Set LCKK bit to enable locking sequence
    GPIOx->LCKR = GPIO_LCKR_LCKK | pin;
    
    // Perform locking sequence
    GPIOx->LCKR = pin;                    // Write 0 to LCKK
    GPIOx->LCKR = GPIO_LCKR_LCKK | pin;   // Write 1 to LCKK
    GPIOx->LCKR = pin;                    // Write 0 to LCKK
    GPIOx->LCKR = GPIO_LCKR_LCKK | pin;   // Write 1 to LCKK
    
    // Read LCKR register
    uint32_t lock_status = GPIOx->LCKR;
    
    // Check if lock was successful
    if (lock_status & GPIO_LCKR_LCKK) {
        return HAL_OK;  // Pin locked successfully
    } else {
        return HAL_ERROR;  // Lock failed
    }
}
```

### Multi-Pin Configuration

```c
// Configure multiple pins at once
void configure_multiple_pins(void) {
    GPIO_InitTypeDef GPIO_InitStruct;
    
    // Configure LED pins (PA0-PA7)
    GPIO_InitStruct.Pin = GPIO_PIN_0 | GPIO_PIN_1 | GPIO_PIN_2 | GPIO_PIN_3 |
                         GPIO_PIN_4 | GPIO_PIN_5 | GPIO_PIN_6 | GPIO_PIN_7;
    GPIO_InitStruct.Mode = GPIO_MODE_OUTPUT_PP;
    GPIO_InitStruct.Pull = GPIO_NOPULL;
    GPIO_InitStruct.Speed = GPIO_SPEED_HIGH;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // Configure button pins (PC0-PC3) with interrupts
    GPIO_InitStruct.Pin = GPIO_PIN_0 | GPIO_PIN_1 | GPIO_PIN_2 | GPIO_PIN_3;
    GPIO_InitStruct.Mode = GPIO_MODE_IT_FALLING;
    GPIO_InitStruct.Pull = GPIO_PULLUP;
    GPIO_InitStruct.Speed = GPIO_SPEED_LOW;
    HAL_GPIO_Init(GPIOC, &GPIO_InitStruct);
    
    // Enable interrupts for all button pins
    HAL_NVIC_SetPriority(EXTI0_IRQn, 0, 0);
    HAL_NVIC_EnableIRQ(EXTI0_IRQn);
    HAL_NVIC_SetPriority(EXTI1_IRQn, 0, 0);
    HAL_NVIC_EnableIRQ(EXTI1_IRQn);
    HAL_NVIC_SetPriority(EXTI2_IRQn, 0, 0);
    HAL_NVIC_EnableIRQ(EXTI2_IRQn);
    HAL_NVIC_SetPriority(EXTI3_IRQn, 0, 0);
    HAL_NVIC_EnableIRQ(EXTI3_IRQn);
}
```

### GPIO Port Operations

```c
// Efficient port-wide operations
void gpio_port_operations(void) {
    // Write entire port
    GPIOA->ODR = 0xAAAA;  // Set alternating bits
    
    // Read entire port
    uint16_t port_value = GPIOA->IDR;
    
    // Toggle entire port (software implementation)
    GPIOA->ODR ^= 0xFFFF;
    
    // Atomic port operations using BSRR
    GPIOA->BSRR = 0xFFFF0000;  // Reset all pins
    GPIOA->BSRR = 0x0000AAAA;  // Set pattern
}

// Port masking for selective operations
void selective_port_operations(uint16_t mask, uint16_t value) {
    // Clear bits in mask
    GPIOA->BSRR = (uint32_t)mask << 16;
    
    // Set bits according to value and mask
    GPIOA->BSRR = (uint32_t)(value & mask);
}
```

### Runtime Pin Reconfiguration

```c
// Dynamically change pin configuration
void reconfigure_pin(GPIO_TypeDef *GPIOx, uint16_t pin, uint32_t new_mode) {
    // Disable interrupts during reconfiguration
    __disable_irq();
    
    // Save current configuration
    uint32_t current_moder = GPIOx->MODER;
    uint32_t current_otyper = GPIOx->OTYPER;
    uint32_t current_ospeedr = GPIOx->OSPEEDR;
    uint32_t current_pupdr = GPIOx->PUPDR;
    
    // Calculate pin bit positions
    uint8_t pin_pos = 0;
    while ((pin & (1 << pin_pos)) == 0) pin_pos++;
    
    // Clear current configuration for this pin
    GPIOx->MODER &= ~(3 << (pin_pos * 2));
    GPIOx->OTYPER &= ~(1 << pin_pos);
    GPIOx->OSPEEDR &= ~(3 << (pin_pos * 2));
    GPIOx->PUPDR &= ~(3 << (pin_pos * 2));
    
    // Apply new configuration
    switch (new_mode) {
        case GPIO_MODE_OUTPUT_PP:
            GPIOx->MODER |= (1 << (pin_pos * 2));
            GPIOx->OTYPER &= ~(1 << pin_pos);
            GPIOx->OSPEEDR |= (3 << (pin_pos * 2));  // High speed
            GPIOx->PUPDR &= ~(3 << (pin_pos * 2));    // No pull
            break;
            
        case GPIO_MODE_INPUT:
            GPIOx->MODER &= ~(3 << (pin_pos * 2));
            GPIOx->PUPDR |= (1 << (pin_pos * 2));     // Pull-up
            break;
            
        case GPIO_MODE_ANALOG:
            GPIOx->MODER |= (3 << (pin_pos * 2));
            break;
    }
    
    // Restore interrupts
    __enable_irq();
}
```

---

## Pin Functions

### STM32F429 Pinout Overview

The STM32F429 has 144 pins in LQFP package with the following distribution:

- **Power Pins**: VDD, VSS, VDDA, VSSA, VREF+, VBAT
- **Oscillator Pins**: HSE (PH0/PH1), LSE (PC14/PC15)
- **JTAG/SWD Pins**: JTMS (PA13), JTCK (PA14), JTDI (PA15), JTDO (PB3), JNTRST (PB4)
- **Reset Pin**: NRST (active low)
- **Boot Pins**: BOOT0, BOOT1
- **GPIO Ports**: 114 I/O pins across GPIOA-GPIOI

### Alternate Function Mapping

| Pin | Default Function | Alternate Functions |
|-----|------------------|-------------------|
| PA0 | GPIO | TIM2_CH1, TIM5_CH1, TIM8_ETR, EVENTOUT, WKUP |
| PA1 | GPIO | TIM2_CH2, TIM5_CH2, EVENTOUT |
| PA2 | GPIO | TIM2_CH3, TIM5_CH3, TIM9_CH1, USART2_TX |
| PA3 | GPIO | TIM2_CH4, TIM5_CH4, TIM9_CH2, USART2_RX |
| PA4 | GPIO | SPI1_NSS, SPI3_NSS, USART2_CK, DCMI_HSYNC, I2S3_WS, EVENTOUT |
| PA5 | GPIO | TIM2_CH1, TIM8_CH1N, SPI1_SCK, I2S1_CK |
| PA6 | GPIO | TIM1_BKIN, TIM3_CH1, TIM8_BKIN, SPI1_MISO, I2S2_MISO |
| PA7 | GPIO | TIM1_CH1N, TIM3_CH2, TIM8_CH1N, SPI1_MOSI, I2S2_MOSI |
| PA8 | GPIO | TIM1_CH1, I2C3_SCL, USART1_CK, OTG_FS_SOF, EVENTOUT |
| PA9 | GPIO | TIM1_CH2, I2C3_SMBA, USART1_TX, DCMI_D0, EVENTOUT |
| PA10 | GPIO | TIM1_CH3, I2C3_SDA, USART1_RX, DCMI_D1, OTG_FS_ID |
| PA11 | GPIO | TIM1_CH4, I2C3_SCL, USART1_CTS, OTG_FS_DM |
| PA12 | GPIO | TIM1_ETR, I2C3_SDA, USART1_RTS, OTG_FS_DP |
| PA13 | JTMS-SWDIO | GPIO |
| PA14 | JTCK-SWCLK | GPIO |
| PA15 | JTDI | TIM2_CH1, TIM8_CH1, SPI1_NSS, SPI3_NSS, I2S3_WS, EVENTOUT |

### System Functions

**JTAG/SWD Interface:**
```c
// Configure JTAG pins (PA13-PA15, PB3-PB4)
void configure_jtag_pins(void) {
    GPIO_InitTypeDef GPIO_InitStruct;
    
    // PA13 (JTMS/SWDIO)
    GPIO_InitStruct.Pin = GPIO_PIN_13;
    GPIO_InitStruct.Mode = GPIO_MODE_AF_PP;
    GPIO_InitStruct.Pull = GPIO_PULLUP;
    GPIO_InitStruct.Speed = GPIO_SPEED_HIGH;
    GPIO_InitStruct.Alternate = GPIO_AF0_SYSTEM;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // PA14 (JTCK/SWCLK)
    GPIO_InitStruct.Pin = GPIO_PIN_14;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // PA15 (JTDI)
    GPIO_InitStruct.Pin = GPIO_PIN_15;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // PB3 (JTDO)
    GPIO_InitStruct.Pin = GPIO_PIN_3;
    HAL_GPIO_Init(GPIOB, &GPIO_InitStruct);
    
    // PB4 (JNTRST)
    GPIO_InitStruct.Pin = GPIO_PIN_4;
    HAL_GPIO_Init(GPIOB, &GPIO_InitStruct);
}
```

**Boot Configuration:**
```c
// Configure boot pins
void configure_boot_pins(void) {
    // BOOT0 pin (PB2 on some packages, dedicated pin on others)
    // BOOT1 pin (PB2 on some packages)
    
    // For STM32F429, BOOT0 is dedicated pin, BOOT1 is PB2
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Pin = GPIO_PIN_2,
        .Mode = GPIO_MODE_INPUT,
        .Pull = GPIO_PULLDOWN,  // Default to boot from flash
        .Speed = GPIO_SPEED_LOW
    };
    HAL_GPIO_Init(GPIOB, &GPIO_InitStruct);
}
```

### Peripheral Pin Configurations

**UART Communication:**
```c
// UART1: PA9 (TX), PA10 (RX)
void configure_uart1_pins(void) {
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Mode = GPIO_MODE_AF_PP,
        .Pull = GPIO_PULLUP,
        .Speed = GPIO_SPEED_HIGH,
        .Alternate = GPIO_AF7_USART1
    };
    
    GPIO_InitStruct.Pin = GPIO_PIN_9;   // TX
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    GPIO_InitStruct.Pin = GPIO_PIN_10;  // RX
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
}

// UART6: PC6 (TX), PC7 (RX)
void configure_uart6_pins(void) {
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Mode = GPIO_MODE_AF_PP,
        .Pull = GPIO_PULLUP,
        .Speed = GPIO_SPEED_HIGH,
        .Alternate = GPIO_AF8_USART6
    };
    
    GPIO_InitStruct.Pin = GPIO_PIN_6;   // TX
    HAL_GPIO_Init(GPIOC, &GPIO_InitStruct);
    
    GPIO_InitStruct.Pin = GPIO_PIN_7;   // RX
    HAL_GPIO_Init(GPIOC, &GPIO_InitStruct);
}
```

**SPI Communication:**
```c
// SPI1: PA5 (SCK), PA6 (MISO), PA7 (MOSI), PA4 (NSS)
void configure_spi1_pins(void) {
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Mode = GPIO_MODE_AF_PP,
        .Pull = GPIO_NOPULL,
        .Speed = GPIO_SPEED_HIGH,
        .Alternate = GPIO_AF5_SPI1
    };
    
    GPIO_InitStruct.Pin = GPIO_PIN_5;   // SCK
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    GPIO_InitStruct.Pin = GPIO_PIN_6;   // MISO
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    GPIO_InitStruct.Pin = GPIO_PIN_7;   // MOSI
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // NSS pin (optional, can be software controlled)
    GPIO_InitStruct.Pin = GPIO_PIN_4;
    GPIO_InitStruct.Mode = GPIO_MODE_OUTPUT_PP;
    GPIO_InitStruct.Pull = GPIO_PULLUP;
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
}
```

**I2C Communication:**
```c
// I2C1: PB6 (SCL), PB7 (SDA)
void configure_i2c1_pins(void) {
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Mode = GPIO_MODE_AF_OD,  // Open-drain for I2C
        .Pull = GPIO_PULLUP,
        .Speed = GPIO_SPEED_HIGH,
        .Alternate = GPIO_AF4_I2C1
    };
    
    GPIO_InitStruct.Pin = GPIO_PIN_6;   // SCL
    HAL_GPIO_Init(GPIOB, &GPIO_InitStruct);
    
    GPIO_InitStruct.Pin = GPIO_PIN_7;   // SDA
    HAL_GPIO_Init(GPIOB, &GPIO_InitStruct);
}
```

---

## Performance Considerations

### GPIO Performance Characteristics

| Parameter | Value | Notes |
|-----------|-------|-------|
| Maximum Toggle Frequency | 180 MHz | Depends on pin and speed setting |
| Output Rise/Fall Time | 2-10 ns | Varies with load capacitance |
| Input Propagation Delay | 1-3 ns | Schmitt trigger response |
| Interrupt Latency | 12-25 cycles | From pin change to ISR |
| Current per Pin | 25 mA max | Total package limit applies |
| Capacitive Load | 50 pF max | For specified timing |

### Optimization Techniques

1. **Speed Configuration**
   ```c
   // Use appropriate speed settings
   GPIO_InitTypeDef GPIO_InitStruct;
   
   // High speed for time-critical signals
   GPIO_InitStruct.Speed = GPIO_SPEED_HIGH;  // 100MHz capability
   
   // Medium speed for general I/O
   GPIO_InitStruct.Speed = GPIO_SPEED_MEDIUM; // 25MHz capability
   
   // Low speed for power saving
   GPIO_InitStruct.Speed = GPIO_SPEED_LOW;   // 2MHz capability
   ```

2. **Atomic Operations**
   ```c
   // Use BSRR for atomic set/clear
   GPIOA->BSRR = GPIO_PIN_5;           // Atomic set
   GPIOA->BSRR = GPIO_PIN_5 << 16;     // Atomic clear
   
   // Avoid read-modify-write for output
   // BAD: GPIOA->ODR ^= GPIO_PIN_5;
   // GOOD: GPIOA->BSRR = GPIO_PIN_5 << ((GPIOA->ODR & GPIO_PIN_5) ? 16 : 0);
   ```

3. **Interrupt Optimization**
   ```c
   // Use direct register access for speed
   void EXTI0_IRQHandler(void) {
       if (EXTI->PR & EXTI_PR_PR0) {
           EXTI->PR = EXTI_PR_PR0;  // Clear interrupt
           
           // Handle interrupt
           GPIOA->BSRR = GPIO_PIN_5;  // Fast response
       }
   }
   ```

### Power Management

```c
// GPIO power optimization
void gpio_power_optimization(void) {
    // Use pull-up/down to prevent floating inputs
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Mode = GPIO_MODE_INPUT,
        .Pull = GPIO_PULLUP,  // Prevent floating
        .Speed = GPIO_SPEED_LOW  // Reduce power
    };
    
    // Configure unused pins as analog to save power
    GPIO_InitStruct.Mode = GPIO_MODE_ANALOG;
    GPIO_InitStruct.Pull = GPIO_NOPULL;
    
    // Disable GPIO clocks when not needed
    __HAL_RCC_GPIOA_CLK_DISABLE();
    __HAL_RCC_GPIOB_CLK_DISABLE();
    // ... disable other GPIO clocks
}

// Wake-up from stop mode using GPIO
void configure_gpio_wakeup(void) {
    // Configure PA0 as wake-up source
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Pin = GPIO_PIN_0,
        .Mode = GPIO_MODE_EVT_RISING,  // Event trigger
        .Pull = GPIO_PULLDOWN
    };
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // Enable wake-up in PWR registers
    // (PWR configuration code would go here)
}
```

### Bus Contention Prevention

```c
// Prevent bus contention in multi-master systems
void prevent_bus_contention(void) {
    // Use open-drain mode for shared signals
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Mode = GPIO_MODE_OUTPUT_OD,
        .Pull = GPIO_PULLUP,
        .Speed = GPIO_SPEED_HIGH
    };
    
    // Configure shared I2C bus
    GPIO_InitStruct.Pin = GPIO_PIN_6 | GPIO_PIN_7;  // SCL, SDA
    GPIO_InitStruct.Mode = GPIO_MODE_AF_OD;
    GPIO_InitStruct.Alternate = GPIO_AF4_I2C1;
    HAL_GPIO_Init(GPIOB, &GPIO_InitStruct);
    
    // Check bus state before transmission
    if (HAL_GPIO_ReadPin(GPIOB, GPIO_PIN_6) == GPIO_PIN_RESET) {
        // Bus is busy, wait or handle error
    }
}
```

---

## Troubleshooting

### Common GPIO Issues

#### 1. Pin Not Working

**Problem**: GPIO pin doesn't respond to read/write operations

**Solutions**:
- Check if GPIO clock is enabled (`__HAL_RCC_GPIOx_CLK_ENABLE()`)
- Verify pin mode configuration (input/output/analog/alternate)
- Check for alternate function conflicts
- Ensure pin is not locked (`GPIOx_LCKR`)
- Verify power supply to the pin
- Check for solder bridges or short circuits

**Debug Code**:
```c
void debug_gpio_pin(GPIO_TypeDef *GPIOx, uint16_t pin) {
    // Check clock enable
    uint32_t ahb1enr = RCC->AHB1ENR;
    if (!(ahb1enr & (1 << (((uint32_t)GPIOx - GPIOA_BASE) / 0x400)))) {
        printf("GPIO clock not enabled\n");
    }
    
    // Check pin configuration
    uint8_t pin_num = 0;
    while ((pin & (1 << pin_num)) == 0) pin_num++;
    
    uint32_t moder = (GPIOx->MODER >> (pin_num * 2)) & 0x3;
    uint32_t otyper = (GPIOx->OTYPER >> pin_num) & 0x1;
    uint32_t ospeedr = (GPIOx->OSPEEDR >> (pin_num * 2)) & 0x3;
    uint32_t pupdr = (GPIOx->PUPDR >> (pin_num * 2)) & 0x3;
    
    printf("Pin %d configuration:\n", pin_num);
    printf("  Mode: %lu\n", moder);
    printf("  Output Type: %lu\n", otyper);
    printf("  Speed: %lu\n", ospeedr);
    printf("  Pull: %lu\n", pupdr);
    
    // Test pin read/write
    if (moder == 1) {  // Output mode
        GPIOx->BSRR = pin;  // Set high
        HAL_Delay(1);
        uint32_t idr_high = GPIOx->IDR & pin;
        
        GPIOx->BSRR = pin << 16;  // Set low
        HAL_Delay(1);
        uint32_t idr_low = GPIOx->IDR & pin;
        
        printf("  Output test: High=%lu, Low=%lu\n", idr_high ? 1 : 0, idr_low ? 1 : 0);
    }
}
```

#### 2. External Interrupt Not Triggering

**Problem**: GPIO interrupt doesn't fire on pin change

**Solutions**:
- Verify interrupt mode configuration (`GPIO_MODE_IT_*`)
- Check SYSCFG EXTICR register for correct port selection
- Ensure EXTI IMR register has interrupt unmasked
- Verify NVIC interrupt is enabled and prioritized
- Check for interrupt pending bit stuck (clear with `EXTI->PR`)
- Confirm pin state actually changes
- Check for noise or glitches on the signal

**Interrupt Debug**:
```c
void debug_exti_interrupt(uint16_t pin) {
    // Check SYSCFG configuration
    uint32_t exti_cr = SYSCFG->EXTICR[pin / 4];
    uint32_t port_sel = (exti_cr >> ((pin % 4) * 4)) & 0xF;
    printf("EXTI port selection: %lu\n", port_sel);
    
    // Check EXTI registers
    printf("EXTI IMR: 0x%08lX\n", EXTI->IMR);
    printf("EXTI EMR: 0x%08lX\n", EXTI->EMR);
    printf("EXTI RTSR: 0x%08lX\n", EXTI->RTSR);
    printf("EXTI FTSR: 0x%08lX\n", EXTI->FTSR);
    printf("EXTI PR: 0x%08lX\n", EXTI->PR);
    
    // Check NVIC
    uint32_t irqn = (pin < 5) ? (EXTI0_IRQn + pin) :
                    (pin < 10) ? EXTI9_5_IRQn : EXTI15_10_IRQn;
    printf("NVIC ISER: 0x%08lX\n", NVIC->ISER[irqn / 32] & (1 << (irqn % 32)));
    
    // Clear any pending interrupt
    EXTI->PR = pin;
}
```

#### 3. Alternate Function Not Working

**Problem**: Peripheral doesn't work on configured GPIO pins

**Solutions**:
- Verify correct alternate function number (`GPIO_AF*`)
- Check AFR register configuration (AFRL/AFRH)
- Ensure pin speed is set appropriately for the peripheral
- Confirm peripheral clock is enabled
- Check for pin conflicts (same pin used by multiple peripherals)
- Verify pin is not in reset state
- Check peripheral-specific requirements (pull-up/down)

**AF Debug**:
```c
void debug_alternate_function(GPIO_TypeDef *GPIOx, uint16_t pin) {
    uint8_t pin_num = 0;
    while ((pin & (1 << pin_num)) == 0) pin_num++;
    
    // Check AFR register
    uint32_t afr = (pin_num < 8) ? GPIOx->AFR[0] : GPIOx->AFR[1];
    uint8_t afr_shift = (pin_num < 8) ? (pin_num * 4) : ((pin_num - 8) * 4);
    uint32_t af_value = (afr >> afr_shift) & 0xF;
    
    printf("Pin %d alternate function: AF%lu\n", pin_num, af_value);
    
    // Check mode
    uint32_t moder = (GPIOx->MODER >> (pin_num * 2)) & 0x3;
    if (moder != 2) {
        printf("ERROR: Pin not in alternate function mode\n");
    }
    
    // List common alternate functions
    const char *af_names[] = {
        "SYSTEM", "TIM1", "TIM3", "TIM8", "I2C1", "SPI1", "SPI3", "USART1",
        "USART6", "I2C2", "OTG_FS", "ETH", "FMC", "DCMI", "LTDC", "EVENTOUT"
    };
    
    if (af_value < 16) {
        printf("Function: %s\n", af_names[af_value]);
    }
}
```

#### 4. High Current Consumption

**Problem**: GPIO pins consuming excessive current

**Solutions**:
- Check for short circuits to power/ground
- Verify output type (push-pull vs open-drain)
- Reduce output speed for lower EMI/power
- Use pull-up/down appropriately to prevent floating
- Disable unused pins or configure as analog
- Check for external load exceeding limits
- Verify voltage levels are correct

**Power Debug**:
```c
void debug_gpio_power_consumption(void) {
    // Check all GPIO configurations
    GPIO_TypeDef *gpio_ports[] = {GPIOA, GPIOB, GPIOC, GPIOD, GPIOE,
                                  GPIOF, GPIOG, GPIOH, GPIOI};
    
    for (int i = 0; i < 9; i++) {
        GPIO_TypeDef *GPIOx = gpio_ports[i];
        
        // Check for floating inputs
        for (int pin = 0; pin < 16; pin++) {
            uint32_t moder = (GPIOx->MODER >> (pin * 2)) & 0x3;
            uint32_t pupdr = (GPIOx->PUPDR >> (pin * 2)) & 0x3;
            
            if (moder == 0 && pupdr == 0) {  // Input, no pull
                printf("WARNING: %c%d floating input\n", 'A' + i, pin);
            }
        }
        
        // Check output configurations
        uint32_t odr = GPIOx->ODR;
        uint32_t moder = GPIOx->MODER;
        
        for (int pin = 0; pin < 16; pin++) {
            uint32_t pin_mode = (moder >> (pin * 2)) & 0x3;
            uint32_t pin_state = (odr >> pin) & 0x1;
            
            if (pin_mode == 1) {  // Output
                // Check for high outputs that might be driving loads
                if (pin_state == 1) {
                    printf("NOTE: %c%d output high\n", 'A' + i, pin);
                }
            }
        }
    }
}
```

#### 5. Signal Integrity Issues

**Problem**: GPIO signals unreliable or noisy

**Solutions**:
- Add decoupling capacitors near power pins
- Use appropriate trace lengths and impedance
- Enable internal pull-up/down resistors
- Reduce output speed to minimize ringing
- Add series resistors for signal integrity
- Use differential signaling for long distances
- Check for ground loops and EMI sources

**Signal Debug**:
```c
void debug_signal_integrity(void) {
    // Test signal stability
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Pin = GPIO_PIN_0,
        .Mode = GPIO_MODE_OUTPUT_PP,
        .Pull = GPIO_NOPULL,
        .Speed = GPIO_SPEED_LOW  // Start with low speed
    };
    HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);
    
    // Toggle pin and check stability
    for (int i = 0; i < 1000; i++) {
        GPIOA->BSRR = GPIO_PIN_0;
        volatile int delay = 10;
        while (--delay);
        
        GPIOA->BSRR = GPIO_PIN_0 << 16;
        delay = 10;
        while (--delay);
        
        // Check if pin state matches expected
        uint32_t expected = (i % 2) ? GPIO_PIN_0 : 0;
        uint32_t actual = GPIOA->IDR & GPIO_PIN_0;
        
        if (actual != expected) {
            printf("Signal integrity issue at iteration %d\n", i);
            break;
        }
    }
    
    printf("Signal integrity test completed\n");
}
```

### Hardware Verification

```c
// Comprehensive GPIO test suite
typedef struct {
    const char *name;
    HAL_StatusTypeDef (*test_func)(void);
} gpio_test_t;

HAL_StatusTypeDef test_gpio_basic_functionality(void) {
    // Test GPIOA Pin 5
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Pin = GPIO_PIN_5,
        .Mode = GPIO_MODE_OUTPUT_PP,
        .Pull = GPIO_NOPULL,
        .Speed = GPIO_SPEED_HIGH
    };
    
    if (HAL_GPIO_Init(GPIOA, &GPIO_InitStruct) != HAL_OK) {
        return HAL_ERROR;
    }
    
    // Test output functionality
    HAL_GPIO_WritePin(GPIOA, GPIO_PIN_5, GPIO_PIN_SET);
    HAL_Delay(1);
    if (HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_5) != GPIO_PIN_SET) {
        return HAL_ERROR;
    }
    
    HAL_GPIO_WritePin(GPIOA, GPIO_PIN_5, GPIO_PIN_RESET);
    HAL_Delay(1);
    if (HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_5) != GPIO_PIN_RESET) {
        return HAL_ERROR;
    }
    
    // Test toggle functionality
    HAL_GPIO_TogglePin(GPIOA, GPIO_PIN_5);
    if (HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_5) != GPIO_PIN_SET) {
        return HAL_ERROR;
    }
    
    return HAL_OK;
}

HAL_StatusTypeDef test_gpio_interrupt_functionality(void) {
    // Configure interrupt pin
    GPIO_InitTypeDef GPIO_InitStruct = {
        .Pin = GPIO_PIN_13,
        .Mode = GPIO_MODE_IT_FALLING,
        .Pull = GPIO_PULLUP
    };
    
    if (HAL_GPIO_Init(GPIOC, &GPIO_InitStruct) != HAL_OK) {
        return HAL_ERROR;
    }
    
    // Enable interrupt
    HAL_NVIC_SetPriority(EXTI15_10_IRQn, 0, 0);
    HAL_NVIC_EnableIRQ(EXTI15_10_IRQn);
    
    // Trigger interrupt by simulating button press
    GPIOC->BSRR = GPIO_PIN_13 << 16;  // Set pin low
    HAL_Delay(1);
    
    // Check if interrupt occurred (would need global flag)
    // This is a simplified test
    
    return HAL_OK;
}

void run_gpio_tests(void) {
    gpio_test_t tests[] = {
        {"Basic GPIO Functionality", test_gpio_basic_functionality},
        {"Interrupt Functionality", test_gpio_interrupt_functionality},
        // Add more tests...
    };
    
    printf("=== GPIO Hardware Test Suite ===\n");
    
    for (int i = 0; i < sizeof(tests)/sizeof(tests[0]); i++) {
        printf("Running %s... ", tests[i].name);
        HAL_StatusTypeDef result = tests[i].test_func();
        printf("%s\n", result == HAL_OK ? "PASS" : "FAIL");
    }
    
    printf("=== Tests Complete ===\n");
}
```

### Performance Analysis

```c
// GPIO performance benchmarking
typedef struct {
    uint32_t toggle_frequency;
    uint32_t interrupt_latency;
    uint32_t read_write_latency;
    uint32_t error_count;
} gpio_performance_t;

void benchmark_gpio_performance(gpio_performance_t *perf) {
    uint32_t start_time, end_time;
    volatile uint32_t dummy;
    
    // Benchmark toggle frequency
    start_time = HAL_GetTick();
    for (int i = 0; i < 100000; i++) {
        GPIOA->BSRR = GPIO_PIN_5;
        GPIOA->BSRR = GPIO_PIN_5 << 16;
    }
    end_time = HAL_GetTick();
    
    perf->toggle_frequency = 100000 * 2 / (end_time - start_time);  // Hz
    
    // Benchmark read/write latency
    start_time = DWT->CYCCNT;  // Use cycle counter
    GPIOA->BSRR = GPIO_PIN_5;
    dummy = GPIOA->IDR;
    end_time = DWT->CYCCNT;
    
    perf->read_write_latency = end_time - start_time;  // CPU cycles
    
    // Benchmark interrupt latency
    // (Would require interrupt handler timing)
    
    printf("GPIO Performance Results:\n");
    printf("Toggle Frequency: %lu kHz\n", perf->toggle_frequency / 1000);
    printf("Read/Write Latency: %lu cycles\n", perf->read_write_latency);
    printf("Errors: %lu\n", perf->error_count);
}
```

---

## Summary

The STM32F429 GPIO peripheral provides comprehensive digital I/O capabilities essential for microcontroller applications. Key takeaways:

### Hardware Features
- **114 I/O Pins**: Organized in 9 ports (A-I) with flexible configuration
- **Multiple Operating Modes**: Input, output, alternate function, and analog
- **Configurable Drive**: Push-pull/open-drain with selectable speed
- **Internal Protection**: Pull-up/down resistors and ESD protection
- **Interrupt Capability**: External interrupt lines for event-driven code
- **Alternate Functions**: Pin multiplexing for peripheral interfaces
- **Bit-Banding**: Atomic operations for thread-safe access

### Software Features
- **HAL-Based API**: Consistent interface across all GPIO operations
- **Flexible Configuration**: Comprehensive pin setup options
- **Interrupt Management**: Configurable trigger edges and priorities
- **Atomic Operations**: BSRR register for race-condition-free updates
- **Lock Mechanism**: Pin configuration protection for security
- **Performance Optimized**: Direct register access for speed-critical code
- **Error Handling**: Robust error detection and reporting

### Performance Characteristics
- **High Speed**: Up to 180MHz toggle frequency
- **Low Latency**: 1-3ns propagation delays
- **Atomic Operations**: Bit-banding for thread safety
- **Interrupt Response**: 12-25 cycle interrupt latency
- **Flexible Drive**: Configurable output strength and speed
- **Power Efficient**: Pull-up/down and speed optimization
- **Robust Design**: ESD protection and current limiting

### Best Practices
1. **Enable Clocks**: Always enable GPIO port clocks before use
2. **Configure Properly**: Set mode, speed, and pull resistors appropriately
3. **Use Atomic Operations**: Employ BSRR for thread-safe pin manipulation
4. **Handle Interrupts**: Clear pending bits and manage priorities
5. **Optimize Power**: Use appropriate speed settings and pull resistors
6. **Check Conflicts**: Verify alternate function assignments
7. **Lock Critical Pins**: Use LCKR for configuration protection
8. **Test Thoroughly**: Verify pin functionality in hardware
9. **Monitor Performance**: Benchmark and optimize critical paths
10. **Handle Errors**: Implement robust error recovery

### Common Applications
- **Digital Control**: LED driving and relay control
- **User Interface**: Button input and status indication
- **Communication**: UART, SPI, I2C pin configuration
- **Sensor Interface**: Digital sensor connections
- **Motor Control**: PWM and direction signals
- **Power Management**: Enable/disable control signals
- **Interrupt Sources**: Event-driven system responses
- **Debug Interface**: JTAG/SWD pin configuration
- **Analog Switching**: ADC/DAC signal routing
- **System Control**: Reset and boot pin management

This tutorial provides comprehensive coverage of GPIO usage on STM32F429, from basic digital I/O to advanced interrupt handling and performance optimization, with detailed troubleshooting and peripheral integration guidance for robust embedded systems.